# WK5 Network Architecture: AI Output Audit

## Purpose
Document the initial AI-generated Docker network architecture, audit it against the networking and security checklist, and preserve test evidence for a demonstration.

## Method
1. Preserve the current configuration and runtime state.
2. Compare the original AI output with subsequent changes.
3. Audit port exposure, subnet isolation, DNS/service discovery, proxy retries, and load balancing.
4. Record observed behavior without correcting flaws during the audit.

## About this notebook

This is **version 1: the documented-flaw notebook**. It audits the Docker network the AI generated, **exactly as the AI wrote it**, and keeps the evidence of what went wrong. Nothing here is corrected; the fixes are in a separate, corrected demo notebook.

| Cell | Meaning |
|---|---|
| **What this cell does** | Plain-English explanation of the code below it |
| Code cell + output | The original audit code and the output it produced on **28 Oct 2026 at 11:33**, kept exactly as recorded |
| **Result & finding** | What the output shows and what it means for the audit |

### Headline findings

1. ❌ **Failover fails.** The AI's `nginx.conf` has no connection timeout and no retry policy (`proxy_next_upstream`). When one backend was stopped, **2 of 5 requests hung and timed out** instead of being passed to the healthy backend (section 2.6). This fails the assignment's "zero dropped requests" requirement.
2. ⚠️ **The health check is misleading.** nginx answers `/healthz` itself with "nginx healthy", even with half the backends down. The backends' own `/healthz` is never used.
3. ❌ **A stopped backend stays down.** `backend1` was stopped for the failover test, and nothing in the AI's design (no `restart:` policy, no health check) detected or recovered it. It was still down when this audit ran.
4. ⚠️ **The running system drifted from the files.** During testing, `internal: true` was added to the private network and deployed, then removed from the file without redeploying. Docker still reports the network as internal; the audited file says it isn't (section 2.5).
5. ✅ **The basics are right.** No backend ports reach the host, nginx is the only dual-homed service, backends are found by service name, and round-robin alternated perfectly while both backends were up.

## The architecture under test

```
                  Host / Internet
                        │
                 localhost:8080
                        │
  ┌───────── public_subnet  (172.20.0.0/16) ─────────┐
  │                                                  │
  │        nginx (load balancer)  172.20.0.2         │
  └────────────────────────┬─────────────────────────┘
                           │   nginx is dual-homed
  ┌───────── private_subnet (172.19.0.0/16) ─────────┐
  │        nginx 172.19.0.4                          │
  │                                                  │
  │   backend1:8000              backend2:8000       │
  │   172.19.0.3                 172.19.0.2          │
  │   ✖ stopped (failover test)  ✔ running          │
  │   no host ports              no host ports       │
  └──────────────────────────────────────────────────┘
```
*Addresses come from `docker network inspect` (section 2.5) and the terminal log (section 2.6).*

| Component | File | Role |
|---|---|---|
| `nginx` | `compose.yaml`, `nginx/nginx.conf` | Reverse proxy, round-robin across `backend_pool`, the only service on both networks |
| `backend1`, `backend2` | `Dockerfile`, `app.py` | Python standard-library web server on port 8000; returns `Request handled by <SERVER_ID>`; has `/healthz` |
| `public_subnet` | `compose.yaml` | Bridge network reachable from the host through nginx's published port 8080 |
| `private_subnet` | `compose.yaml` | Bridge network for the backends, with no published ports |

## Step 1: The AI's initial draft

The AI was asked, acting as a senior network systems architect, for a Docker Compose topology with:

1. Two networks: `public_subnet` (reachable from the host) and `private_subnet` (no direct external port exposure).
2. An Nginx or HAProxy load balancer on **both** networks, exposing port 80/443, with a round-robin upstream pool.
3. At least two backend replicas on `private_subnet` **only**, each returning a unique ID and each with a `/healthz` endpoint.
4. The Compose file, load balancer config, backend code, and run/test instructions.

### Did the draft meet the request?

| Requirement | Status | Evidence |
|---|---|---|
| Two named networks | ✅ Met | `public_subnet` and `private_subnet` (bridge) |
| Backends on the private network only | ✅ Met | `backend1` and `backend2` list only `private_subnet` |
| Load balancer dual-homed | ✅ Met | `nginx` lists both networks |
| Round-robin across ≥ 2 replicas | ✅ Met (implicit) | No algorithm is named, so nginx's default, round-robin, applies |
| Unique ID per instance | ✅ Met | Each backend returns `Request handled by backend1` or `backend2` |
| Backend `/healthz` | ⚠️ Exists but unused | nginx answers `/healthz` itself and never forwards it |
| Exposes 80/443 to the host | ⚠️ Partial | Only `8080 → 80`; no 443/TLS |
| Private network isolated | ⚠️ Partial | No published ports, but no `internal: true` |

**Verdict:** the structure is right on paper. Step 2 checks it against the networking and security checklist and against how it actually behaved.

## Step 2: Audit & verify the AI output

### 2.1 Locate the project and evidence folder

**What this cell does:** sets the project folder to wherever the notebook was opened, creates an `audit_evidence` folder for saved results, and confirms the four project files exist. It does not change any project file.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

PROJECT = Path.cwd()
EVIDENCE = PROJECT / "audit_evidence"
EVIDENCE.mkdir(exist_ok=True)

print("Working directory:", PROJECT)
print("Evidence directory:", EVIDENCE)

for name in ["compose.yaml", "nginx/nginx.conf", "app.py", "Dockerfile"]:
    path = PROJECT / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

Working directory: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5
Evidence directory: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5\audit_evidence
compose.yaml: FOUND
nginx/nginx.conf: FOUND
app.py: FOUND
Dockerfile: FOUND


**Result & finding:** ✅ all four files were found in `...\AD371_APP_ARCH\WK5`, so every later cell reads the real project files. Evidence is saved in `WK5\audit_evidence\`.

### 2.2 Preserve a snapshot of the current state

**What this cell does:**

1. Creates a timestamped folder, `audit_evidence\before_restoration_<date_time>\`.
2. Copies the four project files into it unchanged.
3. Runs `docker compose ps` to record which containers are running, and saves that as `runtime_status.json`.

This freezes the "before" evidence. Each run makes a new folder, so earlier snapshots are never overwritten.

In [2]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess

PROJECT = Path.cwd()
EVIDENCE = PROJECT / "audit_evidence"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SNAPSHOT = EVIDENCE / f"before_restoration_{timestamp}"
SNAPSHOT.mkdir(parents=True, exist_ok=False)

# Preserve the exact current project files.
files = [
    "compose.yaml",
    "nginx/nginx.conf",
    "app.py",
    "Dockerfile",
]

for name in files:
    destination = SNAPSHOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT / name, destination)

# Capture Docker Compose state without changing containers.
result = subprocess.run(
    ["docker", "compose", "ps"],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)

record = {
    "captured_at": datetime.now().isoformat(),
    "project_directory": str(PROJECT),
    "snapshot_directory": str(SNAPSHOT),
    "docker_compose_ps_exit_code": result.returncode,
    "docker_compose_ps_stdout": result.stdout,
    "docker_compose_ps_stderr": result.stderr,
}

(SNAPSHOT / "runtime_status.json").write_text(
    json.dumps(record, indent=2),
    encoding="utf-8",
)

print("Snapshot saved:", SNAPSHOT)
print("\nDocker Compose status:")
print(result.stdout or result.stderr)

Snapshot saved: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5\audit_evidence\before_restoration_20261028_113301

Docker Compose status:
NAME             IMAGE          COMMAND                  SERVICE    CREATED          STATUS          PORTS
wk5-backend2-1   wk5-backend2   "python -u app.py"       backend2   30 minutes ago   Up 30 minutes   8000/tcp
wk5-nginx-1      nginx:alpine   "/docker-entrypoint.â€¦"   nginx      30 minutes ago   Up 30 minutes   0.0.0.0:8080->80/tcp, [::]:8080->80/tcp



**Result & finding:** snapshot `before_restoration_20261028_113301` was saved. The container list shows:

| Container | Status | Ports | Meaning |
|---|---|---|---|
| `wk5-nginx-1` | Up 30 minutes | `0.0.0.0:8080->80/tcp` | ✅ The only service reachable from the host |
| `wk5-backend2-1` | Up 30 minutes | `8000/tcp` (no arrow) | ✅ Port is internal only, not mapped to the host |
| `wk5-backend1-1` | **not listed** | — | ❌ **Not running** |

- ✅ **No port leakage.** An arrow (`->`) means a port is mapped to the host; only nginx has one.
- ❌ **Only one backend is running.** `backend1` was stopped about 30 minutes earlier for the failover test (section 2.6) and never restarted. The AI's design has no `restart:` policy and no health check, so nothing noticed. The running system no longer meets the "at least two replicas" requirement.
- *Display glitch only:* `"/docker-entrypoint.â€¦"` is Windows showing the `…` character with the wrong text encoding. It is not a Docker problem.

### 2.3 Show the preserved configuration

**What this cell does:** prints the saved copies of `compose.yaml` and `nginx.conf` from the snapshot, so the exact configuration being audited is on screen.

In [3]:
from pathlib import Path

print("=== SAVED compose.yaml ===")
print((SNAPSHOT / "compose.yaml").read_text(encoding="utf-8"))

print("\n=== SAVED nginx/nginx.conf ===")
print((SNAPSHOT / "nginx" / "nginx.conf").read_text(encoding="utf-8"))

=== SAVED compose.yaml ===
services:
  nginx:
    image: nginx:alpine
    ports:
      - "8080:80"
    volumes:
      - ./nginx/nginx.conf:/etc/nginx/nginx.conf:ro
    networks:
      - public_subnet
      - private_subnet
    depends_on:
      - backend1
      - backend2

  backend1:
    build: .
    environment:
      SERVER_ID: backend1
    networks:
      - private_subnet

  backend2:
    build: .
    environment:
      SERVER_ID: backend2
    networks:
      - private_subnet

networks:
  public_subnet:
    driver: bridge
  private_subnet:
    driver: bridge


=== SAVED nginx/nginx.conf ===
events {}

http {
    upstream backend_pool {
        server backend1:8000;
        server backend2:8000;
    }

    server {
        listen 80;

        location /healthz {
            access_log off;
            return 200 "nginx healthy\n";
            add_header Content-Type text/plain;
        }

        location / {
            proxy_pass http://backend_pool;
        }
    }
}



**Result & finding:** the lines that matter.

**`compose.yaml`**

| Line(s) | Verdict | Why |
|---|---|---|
| `nginx` → `ports: "8080:80"`, both networks | ✅ | The only published, dual-homed service |
| `backend1` / `backend2` → no `ports:`, only `private_subnet` | ✅ | Backends are not exposed to the host |
| `private_subnet` → `driver: bridge`, no `internal: true` | ⚠️ | Backends can still make outbound internet connections |
| `depends_on` with no `healthcheck:` | ⚠️ | Controls start *order* only, not whether a backend is ready or healthy |
| No `restart:` policy | ❌ | A stopped or crashed backend stays down |

**`nginx.conf`**

| Line(s) | Verdict | Why |
|---|---|---|
| `server backend1:8000; server backend2:8000;` | ✅ | Docker service names, not hardcoded IP addresses |
| No `least_conn` / `ip_hash` | ✅ | nginx's default round-robin; fine for stateless requests (but never stated) |
| No `proxy_connect_timeout` / `proxy_read_timeout` | ❌ | nginx's 60-second defaults apply, so a dead backend can stall a request for up to a minute |
| No `proxy_next_upstream`, `max_fails`, `fail_timeout` | ❌ | Failover runs on untuned defaults |
| No `resolver` | ⚠️ | Backend addresses are looked up once at startup and never refreshed |
| `location /healthz { return 200 "nginx healthy"; }` | ⚠️ | nginx answers health checks itself; the backends are never checked |
| `add_header Content-Type text/plain` in `/healthz` | ⚠️ | Sends a **second** `Content-Type` header (confirmed in section 2.6) |
| No `listen 443` | ⚠️ | No HTTPS/TLS |

### 2.4 Automated checklist checks

**What this cell does:** searches the saved file text for specific settings, records each as `true` or `false`, and saves the result to `step2_static_audit.json`.

> **Limitation:** these are simple text searches of the *files*, not checks of the running system. For example, `nginx_dual_homed` only confirms both network names appear in the file. Section 2.5 checks the live system, and on one point it disagrees.

In [4]:
import json
from datetime import datetime
from pathlib import Path

compose_text = (SNAPSHOT / "compose.yaml").read_text(encoding="utf-8")
nginx_text = (SNAPSHOT / "nginx" / "nginx.conf").read_text(encoding="utf-8")

audit = {
    "captured_at": datetime.now().isoformat(),
    "source_snapshot": str(SNAPSHOT),
    "checks": {
        "backend_ports_published_to_host": (
            "ports:" in compose_text.split("  backend1:", 1)[1].split("  backend2:", 1)[0]
            or "ports:" in compose_text.split("  backend2:", 1)[1].split("networks:", 1)[0]
        ),
        "private_subnet_internal_true": "internal: true" in compose_text,
        "nginx_dual_homed": (
            "      - public_subnet" in compose_text
            and "      - private_subnet" in compose_text
        ),
        "backends_use_private_subnet": (
            "SERVER_ID: backend1" in compose_text
            and "SERVER_ID: backend2" in compose_text
        ),
        "uses_docker_service_names": (
            "server backend1:8000;" in nginx_text
            and "server backend2:8000;" in nginx_text
        ),
        "explicit_proxy_timeouts": (
            "proxy_connect_timeout" in nginx_text
            or "proxy_read_timeout" in nginx_text
        ),
        "proxy_next_upstream_configured": (
            "proxy_next_upstream" in nginx_text
        ),
        "alternative_balancing_algorithm": (
            "least_conn;" in nginx_text or "ip_hash;" in nginx_text
        ),
    },
}

audit_path = EVIDENCE / "step2_static_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print(json.dumps(audit, indent=2))
print("\nAudit saved to:", audit_path)

{
  "captured_at": "2026-10-28T11:36:32.311359",
  "source_snapshot": "C:\\Users\\marky\\Documents\\AD_BAS\\FALL_QTR_27\\AD371_APP_ARCH\\WK5\\audit_evidence\\before_restoration_20261028_113301",
  "checks": {
    "backend_ports_published_to_host": false,
    "private_subnet_internal_true": false,
    "nginx_dual_homed": true,
    "backends_use_private_subnet": true,
    "uses_docker_service_names": true,
    "explicit_proxy_timeouts": false,
    "proxy_next_upstream_configured": false,
    "alternative_balancing_algorithm": false
  }
}

Audit saved to: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5\audit_evidence\step2_static_audit.json


**Result & finding:**

| Check | Recorded | Verdict |
|---|---|---|
| Backend ports published to host | `false` | ✅ **Pass:** no port leakage |
| Private subnet `internal: true` | `false` | ⚠️ Not in the **file** (but see 2.5) |
| nginx dual-homed | `true` | ✅ Both networks present (confirmed in 2.5) |
| Both backends defined | `true` | ✅ |
| Docker service names used | `true` | ✅ **Pass:** no hardcoded IPs |
| Explicit proxy timeouts | `false` | ❌ **Fail** |
| `proxy_next_upstream` configured | `false` | ❌ **Fail** |
| Non-round-robin algorithm | `false` | ✅ Round-robin is in use |

**Main finding:** the AI did not include the connection timeouts or retry settings (`proxy_next_upstream`) that the checklist asks for. Section 2.6 shows what that did in practice.

### 2.5 Inspect the real Docker networks

**What this cell does:**

1. `docker compose config` shows the configuration as written in the files **now**.
2. `docker network inspect` shows the networks Docker **actually** created: their address ranges, whether they are internal, and which containers are attached.

Comparing the two shows whether the running system matches the files. Results are saved to `step2_runtime_network_audit.json`.

In [5]:
import subprocess
import json
from datetime import datetime

def inspect_docker(args):
    result = subprocess.run(
        args,
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    return {
        "command": args,
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr
    }

# Read the network configuration currently declared in Compose.
declared = inspect_docker(["docker", "compose", "config"])

# Inspect the networks that actually exist in Docker.
public_network = inspect_docker(
    ["docker", "network", "inspect", "wk5_public_subnet"]
)
private_network = inspect_docker(
    ["docker", "network", "inspect", "wk5_private_subnet"]
)

network_audit = {
    "captured_at": datetime.now().isoformat(),
    "declared_compose": declared,
    "running_public_network": public_network,
    "running_private_network": private_network
}

output_file = EVIDENCE / "step2_runtime_network_audit.json"
output_file.write_text(
    json.dumps(network_audit, indent=2),
    encoding="utf-8"
)

print("=== DECLARED COMPOSE CONFIGURATION ===")
print(declared["stdout"] or declared["stderr"])

print("\n=== ACTUAL PUBLIC NETWORK ===")
print(public_network["stdout"] or public_network["stderr"])

print("\n=== ACTUAL PRIVATE NETWORK ===")
print(private_network["stdout"] or private_network["stderr"])

print("\nSaved runtime network audit to:", output_file)

=== DECLARED COMPOSE CONFIGURATION ===
name: wk5
services:
  backend1:
    build:
      context: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5
      dockerfile: Dockerfile
    environment:
      SERVER_ID: backend1
    networks:
      private_subnet: null
  backend2:
    build:
      context: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5
      dockerfile: Dockerfile
    environment:
      SERVER_ID: backend2
    networks:
      private_subnet: null
  nginx:
    depends_on:
      backend1:
        condition: service_started
        required: true
      backend2:
        condition: service_started
        required: true
    image: nginx:alpine
    networks:
      private_subnet: null
      public_subnet: null
    ports:
      - mode: ingress
        target: 80
        published: "8080"
        protocol: tcp
    volumes:
      - type: bind
        source: C:\Users\marky\Documents\AD_BAS\FALL_QTR_27\AD371_APP_ARCH\WK5\nginx\nginx.conf
        target: /etc/

**Result & finding:** this is the strongest evidence in the notebook, because it shows the live system rather than the files.

| | `wk5_public_subnet` | `wk5_private_subnet` |
|---|---|---|
| Address range | `172.20.0.0/16` | `172.19.0.0/16` |
| Containers attached | `wk5-nginx-1` only | `wk5-backend2-1`, `wk5-nginx-1` |
| `Internal` | `false` | **`true`** |

**✅ Dual-homing and isolation are correct.** nginx is the only container on both networks, and `backend2` has an address on the private network only, so no backend can be reached from the public side.

**❌ backend1 is missing from the network.** `backend2` is at `.2` and nginx at `.4`. The empty `.3` was `backend1`'s address (confirmed by the terminal log in 2.6) until it was stopped. nginx looked up `backend1` once at startup and has no `resolver` to refresh it, so it keeps sending part of the traffic to that dead address.

**⚠️ Configuration drift.** The files (`docker compose config`) show **no** `internal` setting, but the live private network is `Internal: true`. The terminal log (2.6) explains why: `internal: true` was added and deployed at 11:02 as an isolation test, then removed from `compose.yaml` to restore the AI's original, **without redeploying**. That is why the snapshot folder is named `before_restoration`.

> **Why this matters:** an audit that only read the files (section 2.4) would have reported "not internal" and been wrong about the live system. Always check what is actually running.

### 2.6 Deployment history and failover test (terminal evidence)

These tests were run in the Git Bash terminal **before** this notebook, against the AI's original files. The output below is copied from that terminal session. Times are local (UTC − 7).

| Time | Action | Result |
|---|---|---|
| 10:58 | `docker compose up -d --build` with the AI's original files | 3 containers started; private network **not** internal |
| 10:58 | 10 requests with `curl` | ✅ Perfect round-robin |
| 10:59 | `curl -i /healthz` | 200 from nginx itself; duplicate header |
| 11:02 | Added `internal: true`, then `down` and `up` | Private network recreated as internal; round-robin still ✅ |
| ~11:05 | `docker compose stop backend1`, then 5 requests | ❌ **2 of 5 requests timed out** |
| after | Removed `internal: true` from `compose.yaml` | File restored, but **not redeployed**, causing the drift in 2.5 |
| 11:33 | This notebook was run | Captured backend1 still stopped and the network still internal |

#### Test A: round-robin with both backends up ✅

```
$ for i in $(seq 1 10); do curl -s http://localhost:8080/; echo; done
Request handled by backend2
Request handled by backend1
Request handled by backend2
Request handled by backend1
Request handled by backend2
Request handled by backend1
Request handled by backend2
Request handled by backend1
Request handled by backend2
Request handled by backend1
```
**Finding:** a perfect 5/5 split. Round-robin works and the requests are stateless. ✅ (This run is from after the 11:02 redeploy. The 10:58 run on the original network alternated the same way, starting with `backend1`.)

#### Test B: health check ⚠️

```
$ curl -i http://localhost:8080/healthz
HTTP/1.1 200 OK
Server: nginx/1.31.8
Content-Type: text/plain
Content-Length: 14
Connection: keep-alive
Content-Type: text/plain

nginx healthy
```
**Findings:**
- The reply says **"nginx healthy"** and comes from nginx, not a backend. It would say the same with every backend down.
- `Content-Type` appears **twice**. `add_header` adds a second copy on top of nginx's own. This is harmless in most browsers but is an incorrect response.

#### Test C: failover with backend1 stopped ❌

```
$ docker compose stop backend1
 ✔ Container wk5-backend1-1 Stopped           1.2s

$ for i in $(seq 1 5); do curl -sS --max-time 5 http://localhost:8080/; echo; done
curl: (28) Operation timed out after 5015 milliseconds with 0 bytes received
Request handled by backend2
curl: (28) Operation timed out after 5006 milliseconds with 0 bytes received
Request handled by backend2
Request handled by backend2
```

**Finding: failover fails. 2 of 5 requests (40%) were dropped.**

What happened:
1. Round-robin still sent every other request to `backend1`'s old address, `172.19.0.3`.
2. The container there was stopped, so nothing answered. On a Docker bridge network a connection to a stopped container's address typically gets no reply at all, rather than a fast "refused".
3. With no `proxy_connect_timeout` in the AI's config, nginx would wait up to its **60-second default** before trying `backend2`. The client gave up after 5 seconds.
4. After repeated failures nginx's default passive check (1 failure, then a 10-second pause) briefly stopped using `backend1`, which is consistent with requests 4 and 5 both going to `backend2`.

This directly fails the assignment's QA requirement: *"100% of requests transparently routed to the remaining healthy instance with zero dropped requests."* The cause is exactly the gap found in section 2.4: **no connection timeout and no `proxy_next_upstream` retry policy.**

---
## Findings summary (preserved, not corrected)

| # | Checklist area | Finding | Result | Evidence |
|---|---|---|---|---|
| 1 | Port leakage | Backends have no host port mappings; only nginx publishes `8080:80` | ✅ Pass | 2.2, 2.4 |
| 2 | Dual-homing | nginx is the only container on both networks | ✅ Pass | 2.5 |
| 3 | Service discovery | Upstream uses `backend1` and `backend2` service names, not IPs | ✅ Pass | 2.3, 2.4 |
| 4 | Load balancing | Default round-robin, stateless, perfect 5/5 alternation | ✅ Pass | 2.4, 2.6 A |
| 5 | Timeouts & retries | No connection timeout, no `proxy_next_upstream`, no `max_fails`/`fail_timeout` | ❌ **Fail** | 2.4 |
| 6 | Failover | 2 of 5 requests dropped (5-second timeouts) with one backend stopped | ❌ **Fail** | 2.6 C |
| 7 | Health checks | nginx answers `/healthz` itself; backends are never checked | ⚠️ Gap | 2.6 B |
| 8 | Recovery | No `restart:` policy or health check; stopped backend stayed down | ❌ Fail | 2.2, 2.5 |
| 9 | Service discovery | Backend addresses looked up once at startup (no `resolver`) | ⚠️ Risk | 2.5 |
| 10 | Isolation | `private_subnet` not `internal` in the AI's file | ⚠️ Partial | 2.3, 2.4 |
| 11 | Configuration drift | Live network is internal, but the file is not | ⚠️ Drift | 2.5, 2.6 |
| 12 | Exposure | Only port 80 (via 8080); no 443/TLS | ⚠️ Partial | 2.3 |
| 13 | HTTP correctness | Duplicate `Content-Type` header on `/healthz` | ⚠️ Minor | 2.6 B |

**From reviewing `app.py` and the `Dockerfile`:**
- ✅ The app runs as a non-root user (`USER nobody`) and listens on all interfaces (`0.0.0.0:8000`).
- ✅ Containers stop quickly (1.2 seconds in the terminal log).
- ⚠️ The server handles one request at a time, so one slow request blocks that backend.
- ⚠️ Every path returns 200, including unknown paths such as `/favicon.ico`.

## Conclusion

The AI produced a topology with the right shape. Backends are hidden on a private network with no host ports, a single dual-homed load balancer is the only way in, backends are found by service name, and round-robin balancing worked perfectly while both backends were healthy.

It was **not** a "senior architect" design, because it never planned for failure. With no connection timeout and no retry policy, stopping one backend made **40% of requests time out**. The only health check is answered by the load balancer itself, so it reports healthy regardless of the backends. A stopped backend is never detected or restarted. The audit also showed that the running system had drifted from the files, which only checking the live Docker networks revealed.

These flaws are preserved here as evidence. The corrections are applied and tested in the separate **corrected demo notebook**.